In [ ]:
# mBERT Sakha WordPiece extension

import os, json, regex, torch, re, random
from transformers import BertTokenizer, BertTokenizerFast, AutoModelForTokenClassification

SAKHA_JSON = f"{MAIN_DIR}/sakha-wordpiece-tokenizer-cased.json"
BASE_MODEL = "bert-base-multilingual-cased"
SAVE_DIR   = "mbert_extended_wp_sakha_pieceavg_fair"

SPECIALS = {"[PAD]","[UNK]","[CLS]","[SEP]","[MASK]"}
SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

with open(SAKHA_JSON, "r", encoding="utf-8") as f:
    sj = json.load(f)
sakha_vocab = list(sj["model"]["vocab"].keys())

def is_clean_token(t):
    if not isinstance(t, str) or not t.strip(): return False
    if any(ch.isspace() for ch in t): return False
    if t in SPECIALS: return False
    if regex.fullmatch(r"\p{P}+", t): return False
    if t.isdigit(): return False
    if regex.search(r"\p{C}", t): return False
    if regex.fullmatch(r"(.)\1{3,}", t): return False
    return True

clean_tokens = [t for t in sakha_vocab if is_clean_token(t)]
print("Sakha tokens (clean):", len(clean_tokens))

# --- Prepare a tmp tokenizer dir by appending tokens to vocab.txt ---
tmp_dir = "_tmp_mbert_tok"
os.makedirs(tmp_dir, exist_ok=True)
base_tok = BertTokenizer.from_pretrained(BASE_MODEL, do_lower_case=False)
base_tok.save_pretrained(tmp_dir)

vocab_path = os.path.join(tmp_dir, "vocab.txt")
with open(vocab_path, "r", encoding="utf-8") as f:
    base_vocab = [line.rstrip("\n") for line in f]
base_set = set(base_vocab)

to_add = [t for t in clean_tokens if t not in base_set]
print("Will append to vocab.txt:", len(to_add))

with open(vocab_path, "w", encoding="utf-8") as f:
    f.write("\n".join(base_vocab + to_add) + "\n")

tok = BertTokenizer.from_pretrained(tmp_dir, do_lower_case=False)
print("Vocab size (extended):", len(tok))

model = AutoModelForTokenClassification.from_pretrained(BASE_MODEL, num_labels=7)
old_n = model.get_input_embeddings().weight.size(0)
try:
    _ = model.resize_token_embeddings(len(tok), mean_resizing=False)
except TypeError:
    _ = model.resize_token_embeddings(len(tok))
    print("⚠️ transformers version lacks mean_resizing=False; overwriting new rows manually.")
new_n = model.get_input_embeddings().weight.size(0)
print(f"Resized embeddings: {old_n} -> {new_n}")

with torch.no_grad():
    emb = model.get_input_embeddings().weight

    CYR_RX = re.compile(r"^[\u0400-\u052F\u2DE0-\u2DFF\uA640-\uA69F#\-ʼ’]+$")
    SAKHA_CHARS = set("өүһҕҥӨҮҺҔҢ")

    base_vocab_map = base_tok.get_vocab()
    sakh_ids = [i for tok_, i in base_vocab_map.items()
                if any(ch in SAKHA_CHARS for ch in tok_) and i < old_n]
    cyr_ids  = [i for tok_, i in base_vocab_map.items()
                if tok_ not in SPECIALS and CYR_RX.match(tok_) and i < old_n]

    pool_ids = sakh_ids if len(sakh_ids) >= 500 else (cyr_ids if len(cyr_ids) >= 500 else list(range(old_n)))
    pool_name = "sakha" if pool_ids == sakh_ids else ("cyrillic" if pool_ids == cyr_ids else "global")
    pool = emb[pool_ids]
    pool_mean = pool.mean(dim=0, keepdim=True)
    pool_std  = pool.std(dim=0, keepdim=True).clamp_min(1e-6)
    print(f"Fallback pool: {pool_name} | size={len(pool_ids)}")

    def piece_len(p: str) -> int:
        return max(1, len(p.replace("##", "")))

    init_from_pieces, fallbacks = 0, 0
    for t in to_add:
        new_id = tok.convert_tokens_to_ids(t)
        if new_id is None or new_id < 0:  # safety
            continue
        pieces = base_tok.tokenize(t)

        if len(pieces) == 1 and pieces[0] == "[UNK]":
            emb[new_id] = (pool_mean + 0.02 * torch.randn_like(pool_std) * pool_std).squeeze(0)
            fallbacks += 1
            continue

        ids = [base_tok.convert_tokens_to_ids(p) for p in pieces]
        ids = [i for i in ids if i is not None and 0 <= i < old_n]
        if not ids:
            emb[new_id] = (pool_mean + 0.02 * torch.randn_like(pool_std) * pool_std).squeeze(0)
            fallbacks += 1
            continue
        ws = torch.tensor([piece_len(base_tok.convert_ids_to_tokens(i)) for i in ids],
                          dtype=torch.float32, device=emb.device)
        w = ws / ws.sum()
        emb[new_id] = (emb[ids] * w.unsqueeze(1)).sum(dim=0)
        init_from_pieces += 1

print(f"Initialized from base pieces: {init_from_pieces} | Fallbacks: {fallbacks}")

fast_tok = BertTokenizerFast.from_pretrained(tmp_dir, do_lower_case=False, from_slow=True)

os.makedirs(SAVE_DIR, exist_ok=True)
fast_tok.save_pretrained(SAVE_DIR)
model.save_pretrained(SAVE_DIR)
print("Saved to:", SAVE_DIR)

# Smoke test
sample = "Саха тыл – тылбыра саҥа диэһи."
print("Slow tokens:", tok.tokenize(sample))
print("Fast tokens:", fast_tok.tokenize(sample))
enc = fast_tok(sample, return_offsets_mapping=True)
print("Offsets:", enc["offset_mapping"])


In [ ]:
# ruBERT Sakha WordPiece extension

import os, json, regex, torch, re, random
from transformers import BertTokenizer, BertTokenizerFast, AutoModelForTokenClassification

SAKHA_JSON = f"{MAIN_DIR}/sakha-wordpiece-tokenizer-cased.json"
BASE_MODEL = "ai-forever/ruBert-base"
SAVE_DIR   = "rubert_extended_wp_sakha_pieceavg_fair"

SPECIALS = {"[PAD]","[UNK]","[CLS]","[SEP]","[MASK]"}
SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

with open(SAKHA_JSON, "r", encoding="utf-8") as f:
    sj = json.load(f)
sakha_vocab = list(sj["model"]["vocab"].keys())

def is_clean_token(t):
    if not isinstance(t, str) or not t.strip(): return False
    if any(ch.isspace() for ch in t): return False
    if t in SPECIALS: return False
    if regex.fullmatch(r"\p{P}+", t): return False
    if t.isdigit(): return False
    if regex.search(r"\p{C}", t): return False
    if regex.fullmatch(r"(.)\1{3,}", t): return False
    return True

clean_tokens = [t for t in sakha_vocab if is_clean_token(t)]
print("Sakha tokens (clean):", len(clean_tokens))

tmp_dir = "_tmp_rubert_tok"
os.makedirs(tmp_dir, exist_ok=True)
base_tok = BertTokenizer.from_pretrained(BASE_MODEL, do_lower_case=False)
base_tok.save_pretrained(tmp_dir)

vocab_path = os.path.join(tmp_dir, "vocab.txt")
with open(vocab_path, "r", encoding="utf-8") as f:
    base_vocab = [line.rstrip("\n") for line in f]
base_set = set(base_vocab)

to_add = [t for t in clean_tokens if t not in base_set]
print("Will append to vocab.txt:", len(to_add))

with open(vocab_path, "w", encoding="utf-8") as f:
    f.write("\n".join(base_vocab + to_add) + "\n")

tok = BertTokenizer.from_pretrained(tmp_dir, do_lower_case=False)
print("Vocab size (extended):", len(tok))

model = AutoModelForTokenClassification.from_pretrained(BASE_MODEL, num_labels=7)
old_n = model.get_input_embeddings().weight.size(0)
try:
    _ = model.resize_token_embeddings(len(tok), mean_resizing=False)
except TypeError:
    _ = model.resize_token_embeddings(len(tok))
    print("⚠️ transformers version lacks mean_resizing=False; overwriting new rows manually.")
new_n = model.get_input_embeddings().weight.size(0)
print(f"Resized embeddings: {old_n} -> {new_n}")

with torch.no_grad():
    emb = model.get_input_embeddings().weight

    CYR_RX = re.compile(r"^[\u0400-\u052F\u2DE0-\u2DFF\uA640-\uA69F#\-ʼ’]+$")
    SAKHA_CHARS = set("өүһҕҥӨҮҺҔҢ")

    base_vocab_map = base_tok.get_vocab()
    sakh_ids = [i for tok_, i in base_vocab_map.items()
                if any(ch in SAKHA_CHARS for ch in tok_) and i < old_n]
    cyr_ids  = [i for tok_, i in base_vocab_map.items()
                if tok_ not in SPECIALS and CYR_RX.match(tok_) and i < old_n]

    pool_ids = sakh_ids if len(sakh_ids) >= 500 else (cyr_ids if len(cyr_ids) >= 500 else list(range(old_n)))
    pool_name = "sakha" if pool_ids == sakh_ids else ("cyrillic" if pool_ids == cyr_ids else "global")
    pool = emb[pool_ids]
    pool_mean = pool.mean(dim=0, keepdim=True)
    pool_std  = pool.std(dim=0, keepdim=True).clamp_min(1e-6)
    print(f"Fallback pool: {pool_name} | size={len(pool_ids)}")

    def piece_len(p: str) -> int:
        return max(1, len(p.replace("##", "")))

    init_from_pieces, fallbacks = 0, 0
    for t in to_add:
        new_id = tok.convert_tokens_to_ids(t)
        if new_id is None or new_id < 0:
            continue
        pieces = base_tok.tokenize(t)
        if len(pieces) == 1 and pieces[0] == "[UNK]":
            emb[new_id] = (pool_mean + 0.02 * torch.randn_like(pool_std) * pool_std).squeeze(0)
            fallbacks += 1
            continue
        ids = [base_tok.convert_tokens_to_ids(p) for p in pieces]
        ids = [i for i in ids if i is not None and 0 <= i < old_n]
        if not ids:
            emb[new_id] = (pool_mean + 0.02 * torch.randn_like(pool_std) * pool_std).squeeze(0)
            fallbacks += 1
            continue
        ws = torch.tensor([piece_len(base_tok.convert_ids_to_tokens(i)) for i in ids],
                          dtype=torch.float32, device=emb.device)
        w = ws / ws.sum()
        emb[new_id] = (emb[ids] * w.unsqueeze(1)).sum(dim=0)
        init_from_pieces += 1

print(f"Initialized from base pieces: {init_from_pieces} | Fallbacks: {fallbacks}")

fast_tok = BertTokenizerFast.from_pretrained(tmp_dir, do_lower_case=False, from_slow=True)

os.makedirs(SAVE_DIR, exist_ok=True)
fast_tok.save_pretrained(SAVE_DIR)
model.save_pretrained(SAVE_DIR)
print("Saved to:", SAVE_DIR)

# Smoke test
sample = "Саха тыл – тылбыра саҥа диэһи."
print("Slow tokens:", tok.tokenize(sample))
print("Fast tokens:", fast_tok.tokenize(sample))
enc = fast_tok(sample, return_offsets_mapping=True)
print("Offsets:", enc["offset_mapping"])


In [ ]:

import os, shutil
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForTokenClassification


from google.colab import drive
drive.mount("/content/drive", force_remount=False)


LOCAL_DIR = "rubert_extended_wp_sakha_pieceavg_fair"
DRIVE_DIR = f"{MAIN_DIR}/rubert_extended_wp_sakha_pieceavg_fair"


if os.path.isdir(DRIVE_DIR):
    shutil.rmtree(DRIVE_DIR)
shutil.copytree(LOCAL_DIR, DRIVE_DIR)
print(f"Copied {LOCAL_DIR} -> {DRIVE_DIR}")

tok_chk = AutoTokenizer.from_pretrained(DRIVE_DIR, use_fast=True)
mdl_chk = AutoModelForTokenClassification.from_pretrained(DRIVE_DIR, num_labels=7)

print("Reload OK:")
print("  vocab size:", len(tok_chk))
print("  embedding rows:", mdl_chk.get_input_embeddings().weight.shape[0])
